# Phase 3 — Apply Feature Engineering Pipeline

**Goal:** Apply `FraudFeatureEngineer` to the full IEEE-CIS training data, validate output, and save artifacts for Phase 4 modeling.

**Strategy:**
- **Time-based split** (NOT random!) — train on first ~80% chronologically, validate on last ~20%
- Random splits leak future info into training. Fraud patterns evolve over time; we must validate on later data than we trained on.
- Save outputs as Parquet (faster, smaller than CSV, preserves dtypes)
- Save fitted `FraudFeatureEngineer` as pickle for FastAPI inference in Phase 5

In [1]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add project root to path so we can import src.*
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.features.build_features import FraudFeatureEngineer

# Paths
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

# Ensure output dirs exist
PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Will save processed data to: {PROCESSED}")
print(f"Will save fitted pipeline to: {MODELS_DIR}")

Project root: C:\Projects\fraud-detection-system
Will save processed data to: C:\Projects\fraud-detection-system\data\processed
Will save fitted pipeline to: C:\Projects\fraud-detection-system\models


In [2]:
# Same as Phase 2 — load both files and merge
print("Loading transactions...")
train_txn = pd.read_csv(RAW / "train_transaction.csv")
print(f"  Loaded {len(train_txn):,} rows × {train_txn.shape[1]} columns")

print("Loading identity...")
train_id = pd.read_csv(RAW / "train_identity.csv")
print(f"  Loaded {len(train_id):,} rows × {train_id.shape[1]} columns")

print("Merging on TransactionID...")
df = train_txn.merge(train_id, on="TransactionID", how="left")
print(f"  Merged: {df.shape}")

# Free memory — we don't need the originals anymore
del train_txn, train_id
import gc; gc.collect()

Loading transactions...
  Loaded 590,540 rows × 394 columns
Loading identity...
  Loaded 144,233 rows × 41 columns
Merging on TransactionID...
  Merged: (590540, 434)


0

In [3]:
# Time-based split: train on first 80% chronologically, validate on last 20%
# IMPORTANT: We sort by TransactionDT, NOT random shuffle
df = df.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()

del df; gc.collect()

# Separate target from features
y_train = train_df.pop("isFraud")
y_val = val_df.pop("isFraud")

print(f"Train: {len(train_df):,} rows, fraud rate {y_train.mean()*100:.2f}%")
print(f"Val:   {len(val_df):,} rows, fraud rate {y_val.mean()*100:.2f}%")
print()
print(f"Train date range: {pd.Timestamp('2017-12-01') + pd.to_timedelta(train_df['TransactionDT'].min(), 's')} "
      f"to {pd.Timestamp('2017-12-01') + pd.to_timedelta(train_df['TransactionDT'].max(), 's')}")
print(f"Val date range:   {pd.Timestamp('2017-12-01') + pd.to_timedelta(val_df['TransactionDT'].min(), 's')} "
      f"to {pd.Timestamp('2017-12-01') + pd.to_timedelta(val_df['TransactionDT'].max(), 's')}")

Train: 472,432 rows, fraud rate 3.51%
Val:   118,108 rows, fraud rate 3.44%

Train date range: 2017-12-02 00:00:00 to 2018-04-21 02:54:02
Val date range:   2018-04-21 02:55:00 to 2018-06-01 23:58:51


In [4]:
# Fit on training data only — no leakage from validation
print("Fitting FraudFeatureEngineer on training data...")
fe = FraudFeatureEngineer()
fe.fit(train_df, y_train)
print(f"  Global fraud rate learned: {fe.global_fraud_rate_*100:.4f}%")
print(f"  Email risk map size: {len(fe.email_risk_map_):,} domains")
print(f"  UID amount stats: {len(fe.uid_amt_stats_):,} unique UIDs")
print(f"  UID count map: {len(fe.uid_count_map_):,} unique UIDs")

Fitting FraudFeatureEngineer on training data...
  Global fraud rate learned: 3.5135%
  Email risk map size: 59 domains
  UID amount stats: 82,646 unique UIDs
  UID count map: 82,646 unique UIDs


In [5]:
# Transform both sets
print("Transforming train...")
%time X_train = fe.transform(train_df)
print(f"  Output shape: {X_train.shape}")

print("\nTransforming val...")
%time X_val = fe.transform(val_df)
print(f"  Output shape: {X_val.shape}")

Transforming train...
CPU times: total: 3.53 s
Wall time: 3.57 s
  Output shape: (472432, 36)

Transforming val...
CPU times: total: 828 ms
Wall time: 839 ms
  Output shape: (118108, 36)


In [6]:
# Inspect — what features did we end up with?
print(f"Total features: {X_train.shape[1]}")
print("\nFeature names:")
for col in X_train.columns:
    print(f"  {col:<25} dtype={X_train[col].dtype}")

print("\nSample rows:")
X_train.head()

Total features: 36

Feature names:
  hour                      dtype=int8
  day_of_week               dtype=int8
  hour_sin                  dtype=float32
  hour_cos                  dtype=float32
  dow_sin                   dtype=float32
  dow_cos                   dtype=float32
  is_weekend                dtype=int8
  amt                       dtype=float32
  amt_log                   dtype=float32
  amt_decimal               dtype=float32
  amt_is_round              dtype=int8
  uid_txn_count             dtype=int32
  uid_amt_mean              dtype=float32
  uid_amt_std               dtype=float32
  uid_amt_deviation         dtype=float32
  uid_is_new                dtype=int8
  email_risk                dtype=float32
  email_is_free             dtype=int8
  email_is_missing          dtype=int8
  missing_id_01             dtype=int8
  missing_id_02             dtype=int8
  missing_DeviceType        dtype=int8
  missing_DeviceInfo        dtype=int8
  missing_dist1             dtype=

,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,amt,amt_log,amt_decimal,...,card6,DeviceType,C1,C2,C13,D1,D2,D15,addr1,addr2
0,0,5,0.0,1.0,-0.974928,-0.222521,1,68.5,4.241327,0.5,...,credit,missing,1.0,1.0,1.0,14.0,NaN,0.0,315.0,87.0
1,0,5,0.0,1.0,-0.974928,-0.222521,1,29.0,3.401197,0.0,...,credit,missing,1.0,1.0,1.0,0.0,NaN,0.0,325.0,87.0
2,0,5,0.0,1.0,-0.974928,-0.222521,1,59.0,4.094345,0.0,...,debit,missing,1.0,1.0,1.0,0.0,NaN,315.0,330.0,87.0
3,0,5,0.0,1.0,-0.974928,-0.222521,1,50.0,3.931826,0.0,...,debit,missing,2.0,5.0,25.0,112.0,112.0,111.0,476.0,87.0
4,0,5,0.0,1.0,-0.974928,-0.222521,1,50.0,3.931826,0.0,...,credit,mobile,1.0,1.0,1.0,0.0,NaN,NaN,420.0,87.0


In [7]:
# Sanity check 1: no NaN in engineered numeric features
print("=== NaN counts in engineered features (should be 0) ===")
engineered = ["hour", "hour_sin", "amt_log", "uid_amt_deviation", "email_risk", "is_weekend"]
for col in engineered:
    if col in X_train.columns:
        print(f"  {col:<25} NaN count: {X_train[col].isna().sum()}")

# Sanity check 2: email_risk should range roughly [0, 0.5]
print(f"\nEmail risk range: [{X_train['email_risk'].min():.4f}, {X_train['email_risk'].max():.4f}]")
print(f"Email risk mean:  {X_train['email_risk'].mean():.4f} (should be near global rate {fe.global_fraud_rate_:.4f})")

# Sanity check 3: train/val schemas match exactly
print(f"\nTrain columns == Val columns: {list(X_train.columns) == list(X_val.columns)}")
print(f"Train dtypes == Val dtypes:  {(X_train.dtypes == X_val.dtypes).all()}")

=== NaN counts in engineered features (should be 0) ===
  hour                      NaN count: 0
  hour_sin                  NaN count: 0
  amt_log                   NaN count: 0
  uid_amt_deviation         NaN count: 0
  email_risk                NaN count: 0
  is_weekend                NaN count: 0

Email risk range: [0.0058, 0.2067]
Email risk mean:  0.0360 (should be near global rate 0.0351)

Train columns == Val columns: True
Train dtypes == Val dtypes:  True


In [8]:
# Save engineered datasets as Parquet (compact, typed, fast to reload)
print("Saving processed datasets...")
X_train.to_parquet(PROCESSED / "X_train.parquet", index=False)
y_train.to_frame().to_parquet(PROCESSED / "y_train.parquet", index=False)
X_val.to_parquet(PROCESSED / "X_val.parquet", index=False)
y_val.to_frame().to_parquet(PROCESSED / "y_val.parquet", index=False)

# Save fitted pipeline as pickle for FastAPI inference in Phase 5
print("Saving fitted FraudFeatureEngineer...")
with open(MODELS_DIR / "feature_engineer.pkl", "wb") as f:
    pickle.dump(fe, f)

# Report file sizes
print("\n=== Saved artifacts ===")
for path in [PROCESSED / "X_train.parquet", PROCESSED / "X_val.parquet",
             PROCESSED / "y_train.parquet", PROCESSED / "y_val.parquet",
             MODELS_DIR / "feature_engineer.pkl"]:
    size_mb = path.stat().st_size / 1024**2
    print(f"  {path.relative_to(ROOT)}  ({size_mb:.1f} MB)")

Saving processed datasets...
Saving fitted FraudFeatureEngineer...

=== Saved artifacts ===
  data\processed\X_train.parquet  (11.2 MB)
  data\processed\X_val.parquet  (2.8 MB)
  data\processed\y_train.parquet  (0.0 MB)
  data\processed\y_val.parquet  (0.0 MB)
  models\feature_engineer.pkl  (4.1 MB)


In [9]:
# Round-trip test: load saved pipeline, transform a sample, compare to in-memory output
print("Round-trip verification...")
with open(MODELS_DIR / "feature_engineer.pkl", "rb") as f:
    fe_loaded = pickle.load(f)

# Transform same sample with both pipelines
sample = val_df.head(100)
X_in_memory = fe.transform(sample)
X_from_disk = fe_loaded.transform(sample)

# Should be identical
identical = X_in_memory.equals(X_from_disk)
print(f"  In-memory output == loaded output: {identical}")

if identical:
    print("  ✅ Pipeline serialization works correctly. Phase 5 can load this artifact.")
else:
    print("  ❌ Mismatch detected! Investigate before Phase 4.")
    # Show first diff
    for col in X_in_memory.columns:
        if not X_in_memory[col].equals(X_from_disk[col]):
            print(f"    Column '{col}' differs")

Round-trip verification...
  In-memory output == loaded output: True
  ✅ Pipeline serialization works correctly. Phase 5 can load this artifact.


In [10]:
# Find columns where dtypes differ between train and val
diff_dtypes = []
for col in X_train.columns:
    if X_train[col].dtype != X_val[col].dtype:
        diff_dtypes.append({
            "column": col,
            "train_dtype": str(X_train[col].dtype),
            "val_dtype": str(X_val[col].dtype),
        })

if diff_dtypes:
    print("Columns with differing dtypes:")
    for d in diff_dtypes:
        print(f"  {d['column']:<25} train={d['train_dtype']:<20} val={d['val_dtype']}")
else:
    print("Dtypes match! Investigating categorical category mismatches...")
    for col in X_train.select_dtypes(include="category").columns:
        train_cats = set(X_train[col].cat.categories)
        val_cats = set(X_val[col].cat.categories)
        if train_cats != val_cats:
            print(f"  {col}: categories differ")
            print(f"    train only: {train_cats - val_cats}")
            print(f"    val only:   {val_cats - train_cats}")

Dtypes match! Investigating categorical category mismatches...


In [11]:
# Verify categorical columns now have identical category lists
print("=== Categorical category list match ===")
for col in ["ProductCD", "card4", "card6", "DeviceType"]:
    train_cats = list(X_train[col].cat.categories)
    val_cats = list(X_val[col].cat.categories)
    match = train_cats == val_cats
    print(f"  {col:<15} match={match}  train_cats={train_cats}")

=== Categorical category list match ===
  ProductCD       match=True  train_cats=['C', 'H', 'R', 'S', 'W']
  card4           match=True  train_cats=['american express', 'discover', 'mastercard', 'missing', 'visa']
  card6           match=True  train_cats=['charge card', 'credit', 'debit', 'debit or credit', 'missing']
  DeviceType      match=True  train_cats=['desktop', 'missing', 'mobile']


In [12]:
# Diagnose card6 specifically
print("=== card6 deep inspection ===")
print(f"\nX_train['card6'] dtype: {X_train['card6'].dtype}")
print(f"X_train['card6'].cat.categories: {list(X_train['card6'].cat.categories)}")
print(f"X_train['card6'].cat.codes range: [{X_train['card6'].cat.codes.min()}, {X_train['card6'].cat.codes.max()}]")
print(f"X_train['card6'].value_counts():")
print(X_train['card6'].value_counts())

print(f"\nX_val['card6'] dtype: {X_val['card6'].dtype}")
print(f"X_val['card6'].cat.categories: {list(X_val['card6'].cat.categories)}")
print(f"X_val['card6'].cat.codes range: [{X_val['card6'].cat.codes.min()}, {X_val['card6'].cat.codes.max()}]")
print(f"X_val['card6'].value_counts():")
print(X_val['card6'].value_counts())

# The fitted pipeline's stored categories
import pickle
with open(MODELS_DIR / "feature_engineer.pkl", "rb") as f:
    fe_loaded = pickle.load(f)
print(f"\nfe.cat_categories_['card6']: {fe_loaded.cat_categories_['card6']}")

=== card6 deep inspection ===

X_train['card6'] dtype: category
X_train['card6'].cat.categories: ['charge card', 'credit', 'debit', 'debit or credit', 'missing']
X_train['card6'].cat.codes range: [0, 4]
X_train['card6'].value_counts():
card6
debit              349843
credit             121716
missing               828
debit or credit        30
charge card            15
Name: count, dtype: int64

X_val['card6'] dtype: category
X_val['card6'].cat.categories: ['charge card', 'credit', 'debit', 'debit or credit', 'missing']
X_val['card6'].cat.codes range: [1, 4]
X_val['card6'].value_counts():
card6
debit              90095
credit             27270
missing              743
charge card            0
debit or credit        0
Name: count, dtype: int64

fe.cat_categories_['card6']: ['charge card', 'credit', 'debit', 'debit or credit', 'missing']


In [13]:
hasattr(fe, "cat_categories_")

True

In [14]:
# Restore full category lists after Parquet round-trip
# (Parquet prunes unused categories from category dtype columns)
print("=== Restoring categorical dtypes from fitted pipeline ===")
for col, cats in fe.cat_categories_.items():
    if col in X_train.columns:
        X_train[col] = pd.Categorical(X_train[col].astype("object"), categories=cats)
    if col in X_val.columns:
        X_val[col] = pd.Categorical(X_val[col].astype("object"), categories=cats)

# Re-save with full category lists preserved
X_train.to_parquet(PROCESSED / "X_train.parquet", index=False)
X_val.to_parquet(PROCESSED / "X_val.parquet", index=False)
print("  Re-saved X_train.parquet and X_val.parquet with restored categoricals")

# Final verification
print("\n=== Final category list check ===")
for col in fe.cat_categories_:
    train_cats = list(X_train[col].cat.categories)
    val_cats = list(X_val[col].cat.categories)
    print(f"  {col:<15} train=={val_cats==train_cats}  cats={train_cats}")

print(f"\nDtypes match: {(X_train.dtypes == X_val.dtypes).all()}")

=== Restoring categorical dtypes from fitted pipeline ===
  Re-saved X_train.parquet and X_val.parquet with restored categoricals

=== Final category list check ===
  ProductCD       train==True  cats=['C', 'H', 'R', 'S', 'W']
  card4           train==True  cats=['american express', 'discover', 'mastercard', 'missing', 'visa']
  card6           train==True  cats=['charge card', 'credit', 'debit', 'debit or credit', 'missing']
  DeviceType      train==True  cats=['desktop', 'missing', 'mobile']

Dtypes match: True
